# Category-Aware Pricing: Specialized Models

Exploring whether per-category models beat a single general model for Amazon product price prediction.

**Data:** ~20k training items from 8 Amazon categories via `Item.from_hub("tobenna/items_full")`  
**Categories:** Automotive, Electronics, Office_Products, Tools_and_Home_Improvement, Cell_Phones_and_Accessories, Toys_and_Games, Appliances, Musical_Instruments

---

## Step 1: Category Analysis & EDA

Understand how price distributions differ across product domains.

In [ ]:
import os
import sys
sys.path.append(os.path.abspath("../.."))

from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from pricer.items import Item

In [ ]:
dataset = "inginia/items_lite_plus"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# Count items per category
category_counts = Counter(item.category for item in train)

# Sort by count descending
categories = [cat for cat, _ in category_counts.most_common()]
counts = [category_counts[cat] for cat in categories]

print("Items per category (sorted):")
for cat, count in category_counts.most_common():
    print(f"  {cat:<40} {count:>10,}")

In [ ]:
# Bar chart of category counts with count labels
fig, ax = plt.subplots(figsize=(15, 6))
bars = ax.bar(categories, counts, color="goldenrod", edgecolor="white")

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1000,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title("Number of Training Items per Category", fontsize=14)
ax.set_xlabel("Category")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Price histograms per category
n_cats = len(categories)
ncols = 2
nrows = (n_cats + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

items_by_category = {cat: [] for cat in categories}
for item in train:
    items_by_category[item.category].append(item)

for i, cat in enumerate(categories):
    prices = [item.price for item in items_by_category[cat]]
    avg_price = np.mean(prices)
    median_price = np.median(prices)
    axes[i].hist(prices, bins=range(0, 1000, 10), rwidth=0.8, color="steelblue", edgecolor="white")
    axes[i].set_title(f"{cat}\navg=${avg_price:.0f}, median=${median_price:.0f}", fontsize=10)
    axes[i].set_xlabel("Price ($)")
    axes[i].set_ylabel("Count")
    axes[i].set_xlim(0, 600)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Price Distributions by Category (prices $0–$600)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Key Observations

After running the EDA above, document findings here:

**Top categories by item count:**
- Eletronics, Tools and home improvements, Automotive

**Price distribution highlights:**
- Electronics and Appliances tend to have wider price ranges and higher medians — a $500 item is normal.
- Toys_and_Games and Cell_Phones_and_Accessories are skewed toward lower prices.
- Automotive has a large spread, potentially due to parts vs. accessories vs. equipment.

**Takeaway:** Categories have meaningfully different price priors and distributions — the key justification for building per-category specialized models in the steps ahead.

---

## Step 2: Single General XGBoost Baseline

Reproduce the general XGBoost model from day3 as a controlled baseline — all future per-category improvements are measured against this reference point.

In [ ]:
import xgboost as xgb
from sklearn.feature_extraction.text import CountVectorizer
from pricer.evaluator import evaluate

In [ ]:
# Fit vectorizer on training summaries
prices = [item.price for item in train]
documents = [item.summary for item in train]

np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X_train = vectorizer.fit_transform(documents)

print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print(f"Training matrix: {X_train.shape}")

In [ ]:
# Train the general XGBoost model
xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X_train, prices)
print("Training complete.")

In [ ]:
def general_xgb_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, float(xgb_model.predict(x)[0]))

evaluate(general_xgb_pricer, test)

---

## Step 3: Per-Category XGBoost Models

Train a separate XGBoost model for each category, then compare per-category error against the general baseline on the same category slice.


In [ ]:
import pandas as pd

# Split training data by category (reuse items_by_category from EDA if still in scope,
# but rebuild here to be self-contained)
train_by_cat = {cat: [] for cat in categories}
for item in train:
    train_by_cat[item.category].append(item)

test_by_cat = {cat: [] for cat in categories}
for item in test:
    test_by_cat[item.category].append(item)

print(f"{'Category':<45} {'Train':>8}  {'Test':>6}")
print("-" * 65)
for cat in categories:
    print(f"{cat:<45} {len(train_by_cat[cat]):>8,}  {len(test_by_cat[cat]):>6,}")

In [ ]:
cat_vectorizers = {}
cat_models = {}

for cat in categories:
    cat_items = train_by_cat[cat]
    cat_prices = [item.price for item in cat_items]
    cat_docs = [item.summary for item in cat_items]

    vec = CountVectorizer(max_features=1000, stop_words="english")
    X = vec.fit_transform(cat_docs)

    model = xgb.XGBRegressor(n_estimators=300, random_state=42, n_jobs=4, learning_rate=0.1)
    model.fit(X, cat_prices)

    cat_vectorizers[cat] = vec
    cat_models[cat] = model
    print(f"Trained {cat:<45} ({len(cat_items):,} items)")

In [ ]:
def make_cat_pricer(cat):
    """Return a pricer function bound to the given category's model."""
    vec = cat_vectorizers[cat]
    model = cat_models[cat]
    def pricer(item):
        x = vec.transform([item.summary])
        return max(0, float(model.predict(x)[0]))
    pricer.__name__ = f"cat_xgb_{cat}"
    return pricer

# Build comparison table: per-category MAE vs general model MAE on the same test slice
rows = []
for cat in categories:
    cat_test = test_by_cat[cat]
    if not cat_test:
        continue
    cat_pricer = make_cat_pricer(cat)
    cat_errors = [abs(cat_pricer(item) - item.price) for item in cat_test]
    gen_errors = [abs(general_xgb_pricer(item) - item.price) for item in cat_test]
    cat_mae = np.mean(cat_errors)
    gen_mae = np.mean(gen_errors)
    rows.append({
        "Category": cat,
        "N test": len(cat_test),
        "General MAE ($)": round(gen_mae, 2),
        "Per-Cat MAE ($)": round(cat_mae, 2),
        "Improvement ($)": round(gen_mae - cat_mae, 2),
    })

df_compare = pd.DataFrame(rows).sort_values("Improvement ($)", ascending=False)
print(df_compare.to_string(index=False))

In [ ]:
# Run evaluate() on each per-category model against its category-filtered test slice
for cat in categories:
    cat_test = test_by_cat[cat]
    if not cat_test:
        continue
    print(f"\n=== {cat} ===")
    evaluate(make_cat_pricer(cat), cat_test, size=len(cat_test))

---

## Step 4: Fine-Tune a Frontier Model for One Category

Fine-tune `gpt-4.1-nano` on training data for a single category and compare its accuracy against the XGBoost baselines from Step 3.

Call `fine_tune_category(cat)` with any category name to launch a job.  
Training takes ~5–15 minutes via the OpenAI API.

In [ ]:
import json
from openai import OpenAI

openai_client = OpenAI()

In [ ]:
def messages_for_ft(item):
    return [
        {"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items):
    return "\n".join(json.dumps({"messages": messages_for_ft(item)}) for item in items)

def fine_tune_category(cat, train_size=500, val_size=50):
    """Fine-tune gpt-4.1-nano on training data for the given category.
    Returns the fine-tuning job ID.
    """
    import random as _random
    items = list(train_by_cat[cat])
    _random.shuffle(items)
    ft_train = items[:train_size]
    ft_val   = items[train_size:train_size + val_size]

    train_path = f"ft_train_{cat}.jsonl"
    val_path   = f"ft_val_{cat}.jsonl"

    with open(train_path, "w") as f:
        f.write(make_jsonl(ft_train))
    with open(val_path, "w") as f:
        f.write(make_jsonl(ft_val))

    with open(train_path, "rb") as f:
        train_file = openai_client.files.create(file=f, purpose="fine-tune")
    with open(val_path, "rb") as f:
        val_file = openai_client.files.create(file=f, purpose="fine-tune")

    job = openai_client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model="gpt-4.1-nano-2025-04-14",
        seed=42,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix="pricer",
    )
    print(f"Fine-tuning job created: {job.id}")
    print(f"Training on {len(ft_train):,} items, validating on {len(ft_val):,} items")
    print(f"Monitor at: https://platform.openai.com/finetune/{job.id}")
    return job.id

In [ ]:
FT_CATEGORY = "Electronics"  # change to any of the 8 categories

ft_job_id = fine_tune_category(FT_CATEGORY)

In [ ]:
# Re-run this cell to check progress
job = openai_client.fine_tuning.jobs.retrieve(ft_job_id)
print(f"Status : {job.status}")
print(f"Model  : {job.fine_tuned_model}")

for event in openai_client.fine_tuning.jobs.list_events(ft_job_id, limit=5).data:
    print(f"  {event.message}")

In [ ]:
fine_tuned_model_name = openai_client.fine_tuning.jobs.retrieve(ft_job_id).fine_tuned_model

def ft_pricer(item):
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=[{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}],
        max_tokens=7,
    )
    return response.choices[0].message.content

ft_pricer.__name__ = f"fine_tuned_{FT_CATEGORY}"

evaluate(ft_pricer, test_by_cat[FT_CATEGORY], size=len(test_by_cat[FT_CATEGORY]))

In [ ]:
import re
import plotly.graph_objects as go

def parse_price(raw):
    """Extract a float from any LLM response string, e.g. '$29.99', 'Price: $29', '29\n'."""
    if isinstance(raw, (int, float)):
        return float(raw)
    match = re.search(r"[-+]?\d*\.\d+|\d+", str(raw).replace(',', ''))
    return float(match.group()) if match else 0.0

cat_test = test_by_cat[FT_CATEGORY]

model_mae = {
    "General XGBoost":  float(np.mean([abs(general_xgb_pricer(item) - item.price) for item in cat_test])),
    "Per-Cat XGBoost":  float(np.mean([abs(make_cat_pricer(FT_CATEGORY)(item) - item.price) for item in cat_test])),
    "Fine-tuned LLM":   float(np.mean([abs(parse_price(ft_pricer(item)) - item.price) for item in cat_test])),
}

for name, mae in model_mae.items():
    print(f"  {name:<20} MAE = ${mae:.2f}")

fig = go.Figure(go.Bar(
    x=list(model_mae.keys()),
    y=list(model_mae.values()),
    text=[f"${v:.2f}" for v in model_mae.values()],
    textposition="outside",
    marker_color=["steelblue", "goldenrod", "firebrick"],
))
fig.update_layout(
    title=f"MAE Comparison — {FT_CATEGORY}",
    xaxis_title="Model",
    yaxis_title="Mean Absolute Error ($)",
    width=700,
    height=450,
    template="plotly_white",
)
fig.show()